In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from matplotlib import pyplot as plt
import matplotlib
import statsmodels
import sklearn
import xgboost
import torch
import time
import helper_functions as hf

In [ ]:
print(f'Pandas version: {pd.__version__}')
print(f'Numpy version: {np.__version__}')
print(f'Seaborn version: {sns.__version__}')
print(f'Matplotlib version: {matplotlib.__version__}')
print(f'Statsmodels version: {statsmodels.__version__}')
print(f'Scikit-learn version: {sklearn.__version__}')
print(f'XGBoost version: {xgboost.__version__}')
print(f'Torch version: {torch.__version__}')

In [ ]:
global_start_time = time.time()
try:
	customer_data = pd.read_csv('../data/Lending Club Loan Data/X.csv')
	target = pd.read_csv('../data/Lending Club Loan Data/target.csv')
	customer_data['target'] = target['y']
	pd.options.display.max_columns = None
	pd.options.display.max_rows = 100
except Exception as e:
	print(f"An error occurred while loading the data: '{e}'")

In [ ]:
customer_data.head()

In [ ]:
customer_data.columns

### We have 2 139 643 records and 46 columns

In [ ]:
customer_data.shape

Data is inbalanced, around 13 % of data refer to the default case (1), other refer to non-default case (0)

In [ ]:
from sklearn.model_selection import train_test_split

""" Reducing the dataset to 2% of its original size while maintaining the class distribution """
random_state = 68
customer_data, _ = train_test_split(
	customer_data,
	train_size=0.02,
	stratify=customer_data['target'],
	random_state=random_state
)
customer_data = customer_data.reset_index(drop=True)
customer_data['target'].value_counts()

### Handling columns with many NaN values

In [ ]:
nans_count_per_feat = pd.DataFrame(
    data=customer_data.isna().sum().sort_values(ascending=False),
    index=customer_data.isna().sum().sort_values(ascending=False).index,
    columns=['Missing count']
)
nans_count_per_feat['Feature'] = nans_count_per_feat.index
nans_count_per_feat = nans_count_per_feat.reset_index(drop=True)
nans_count_per_feat = nans_count_per_feat[['Feature', 'Missing count']]
nans_count_per_feat[nans_count_per_feat['Missing count'] > 0]

In [ ]:
import os 

""" Saving results in pd.DataFrame to LaTeX tables"""
def save_latex_table(df, cwd_rel_path, include_index=False, float_format="%.2f"):
	# Get project folder path, going one level up with dirname() because this notebook is in src folder
	cwd = os.path.dirname(os.getcwd()) 
	# Convert the DataFrame to LaTeX format and save it to a file
	df_latex = (
		df
		.to_latex(index=include_index, float_format=float_format)
		# Replace underscores with spaces for better readability in LaTeX tables and to avoid LaTeX treating underscores as lower indexing
		.replace('_', ' ')  
	)
	file_path = os.path.join(cwd, cwd_rel_path)
	with open(file_path, 'w') as f:
		f.write(df_latex)

save_latex_table(
	df=nans_count_per_feat[nans_count_per_feat['Missing count'] > 0],
	cwd_rel_path='latex_tables/missing_values.tex',
	include_index=False,
	float_format="%.0f"
 )

In [ ]:
""" 
Filling missing values in 'emp_title' and 'emp_length' with 'unknown' value to indicate missing employment information. 
Both of the variables will be treated as categorical featues.
"""
customer_data[['emp_title', 'emp_length']] = (
    customer_data[['emp_title', 'emp_length']].fillna('unknown')
)

In [ ]:
"""
Analysis for 'mths_since_delinq_...' like columns, because the NaNs in those columns may indicate that the customer has never been 
delinquent, which is a valuable information.
"""
mths_since_like_cols = [
	'mths_since_last_delinq',
	'mths_since_recent_bankcard_delinq',
	'mths_since_recent_revol_delinq'
]
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(20, 5))
for i, col in enumerate(mths_since_like_cols):
	nan_cases_per_target = customer_data[ customer_data[col].isna() ]['target'].value_counts()
	barplot = sns.barplot(
		x=nan_cases_per_target.index, 
		y=nan_cases_per_target.values, 
		ax=axes[i],
		edgecolor='black',
		color='darkgrey'
	)
	for container in barplot.containers:
		barplot.bar_label(container)
	axes[i].set_xlabel('Target')
	axes[i].set_ylabel('NaN cases count')
	axes[i].set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
"""
The mths_since_delinq_... like columns have a significant number of NaN values, and the majority of those NaN values belong to the customers with target=0 (non-defaulters).
This suggests that the NaN values in these columns likely indicate that the customer has never been delinquent, which is a valuable information for the credit scoring model.
So we will fill missing values in the columns ('mths_since_last_delinq', 'mths_since_recent_bankcard_delinq', 'mths_since_recent_revol_delinq') with -1 (to indicate this special case)
"""
customer_data[mths_since_like_cols] = (
    customer_data[mths_since_like_cols].fillna(-1)
)


In [ ]:
"""
Adding binary flag columns to indicate whether the original 'mths_since_delinq_...' like column had missing values (which we have
filled with -1). At the same time reorganizing the columns to place the new flag columns next to their corresponding original columns.
"""
current_cols = customer_data.columns.tolist()
new_cols_order = []
for col in current_cols:
    if col in mths_since_like_cols:
        customer_data[f'{col}_flag'] = np.where(customer_data[col] == -1, 0, 1)
        new_cols_order.extend([col, f'{col}_flag'])
    else:
        new_cols_order.append(col)
        
customer_data = customer_data[new_cols_order]
customer_data.head()

In [ ]:
""" 
Other columns that have a significant number of missing values where those 
missing values cannot be unequivocally interpreted are removed
"""
major_nans_in_columns = (
    customer_data
    .isna()
    .sum() > 0.3 * len(customer_data)
)
customer_data = customer_data.loc[:, ~major_nans_in_columns]

In [ ]:
""" No NaNs at this stage """
any( (customer_data.isna().values.flatten()) )

In [ ]:
customer_data = (
	customer_data
	.dropna()
	.reset_index(drop=True)
)
customer_data.shape

In [ ]:
customer_data.head()

In [ ]:
(customer_data['funded_amnt'].value_counts().index.min(),
customer_data['funded_amnt'].value_counts().index.max())

In [ ]:
customer_data.info()

In [ ]:
""" Inserting new feature - credit_history_age column based on the appropriate date columns """
try:
	credit_history_col_index = (
		customer_data
		.columns
		.to_list()
		.index('issue_date_year') + 1
	)
	customer_data.insert(
		credit_history_col_index,
		'credit_history_age',
		customer_data['issue_date_year'] - customer_data['earliest_cr_line_year']
	)
	# removing the columns related to credit history column
	customer_data = customer_data.drop(
		columns=['issue_date_month', 'issue_date_year', 'earliest_cr_line_month', 'earliest_cr_line_year']
	)
except Exception as e:
	print(f"The 'credit_history' column has been inserted already: '{e}'")

In [ ]:
""" Save flag cols to exclude them from numerical describe table, because those are binary values """
flag_col_mask = customer_data.columns.str.contains('_flag')
flag_cols = customer_data.columns[flag_col_mask]
flag_cols

In [ ]:
""" Excluding flag cols and target col from numeric describe table, because they are binary """
numerical_data, numerical_data_describe = hf.numeric_describe(
    customer_data[ customer_data.columns.difference([*flag_cols, 'target']) ]
)
numerical_data_describe

"loan_term_months" column has only 2 values so we can treat it as category variable, the "region_code" variable is also converted to object

In [ ]:
customer_data['loan_term_months'] = customer_data['loan_term_months'].astype('object')
customer_data['region_code'] = customer_data['region_code'].astype('object')

In [ ]:
""" 
Adding log-transformed columns for annual_income, late_fees_rec and total_credit_revolving_bal, because of high 
variability and extremely high positive skewness, log(x+1) transformation is applied to avoid 0 inside log 
"""
try:
	ann_inc_idx = customer_data.columns.to_list().index('annual_income')
	customer_data.insert(
		ann_inc_idx + 1, 
		'log_annual_income', 
		np.log(customer_data['annual_income'] + 1)
	)
	
	interest_rec_idx = customer_data.columns.to_list().index('interest_rec')
	customer_data.insert(
		interest_rec_idx + 1, 
		'log_interest_rec', 
		np.log(customer_data['interest_rec'] + 1)
	)

	late_fees_rec_idx = customer_data.columns.to_list().index('late_fees_rec')
	customer_data.insert(
		late_fees_rec_idx + 1, 
		'log_late_fees_rec', 
		np.log(customer_data['late_fees_rec'] + 1)
	)
	
	remaining_princ_for_tot_amnt_fund_idx = customer_data.columns.to_list().index('remaining_princ_for_tot_amnt_fund')
	customer_data.insert(
		remaining_princ_for_tot_amnt_fund_idx + 1, 
		'log_remaining_princ_for_tot_amnt_fund', 
		np.log(customer_data['remaining_princ_for_tot_amnt_fund'] + 1)
	)
	
	tot_credit_revol_bal_idx = customer_data.columns.to_list().index('total_credit_revolving_bal')
	customer_data.insert(
		tot_credit_revol_bal_idx + 1, 
		'log_total_credit_revolving_bal', 
		np.log(customer_data['total_credit_revolving_bal'] + 1)
	)
except Exception as e:
	print(f"The log-transformed columns have been inserted already: '{e}'")

In [ ]:
customer_data.head()

In [ ]:
""" Numerical describe including the log transformed variables """
numerical_data, numerical_data_describe = hf.numeric_describe(
    customer_data[ customer_data.columns.difference([*flag_cols, 'target']) ]
)
numerical_data_describe

In [ ]:
# !pip install fpdf
hf.dataframe_to_pdf(
    numerical_data_describe.loc[:, ~numerical_data_describe.columns.isin(['count', '25%', '50%', '75%'])], 
    rel_path='pdf_outputs/numerical_data_describe.pdf', 
    title="Numerical data describe (statistics)", 
    max_rows=30,
    max_table_width=190
)

In [ ]:
categorical_data = (
    customer_data
    .select_dtypes(['object', 'category'])
    .dropna()
    .reset_index(drop=True)
)
categorical_data.describe().transpose().rename(columns={'top': 'most frequent category'})

In [ ]:
(numerical_data.columns.size, categorical_data.columns.size)

### Data Visualization

In [ ]:
hf.draw_distribution(
    df=customer_data,
    subplot_size=(23, 85),
    subplot_cols=3,
	plot_color='darkgrey',
    title_fontsize=16
)

In [ ]:
categorical_data = customer_data.select_dtypes(['object', 'category'])
categorical_data.head()

Categorical and numerical statistics (after adjusting variables to the proper groups of data)

In [ ]:
categorical_data.describe().transpose()

In [ ]:
numerical_data.columns

In [ ]:
hf.draw_boxplots(
    df=numerical_data, # all numerical columns excluding the binary ones like the '_flag' columns and target
    subplot_size=(15, 33),
    plot_color='darkgrey'
)

In [ ]:
"""
Checking data after removing the outliers, but it turns out that then we lose minority class so the outliers 
will be kept, because they seem to strongly represent the minority class
"""
df_dropped_outliers = hf.handle_outliers(customer_data, remove=True)
df_dropped_outliers['target'].value_counts()

In [ ]:
categorical_data.describe().transpose()

In [ ]:
cat_data_df_describe = categorical_data.describe().transpose()
hf.dataframe_to_pdf(
	cat_data_df_describe, 
	'pdf_outputs/categorical_data_describe.pdf', 
	title="Categorical data describe (statistics)", 
	max_rows=30,
    max_table_width=110,
    int_values=True
)

In [ ]:
""" Extracting grades in a more general form (only the letters ranging from A to G), avoiding too many unique values """
# customer_data['grade'] = customer_data['grade'].str[0]

In [ ]:
""" Dropping the columns with too many unique values """
try:
	customer_data = customer_data.drop(
		columns=['grade', 'emp_title', 'addr_state']
	)
except Exception as e:
	print(f"The categorical columns with too many unique values have been already removed: {e}")
customer_data.select_dtypes('object').describe().transpose()

In [ ]:
""" Dropping the original columns for which the log(x+1) transform column was added """
customer_data = customer_data.drop(
    columns=['annual_income', 'interest_rec', 'late_fees_rec', 'remaining_princ_for_tot_amnt_fund', 'total_credit_revolving_bal']
)

In [ ]:
customer_data.head()

In [ ]:
customer_data.shape

In [ ]:
""" Splitting data into training (80 %) and test (20 %) sets """
df_train, df_test = train_test_split(
	customer_data,
    train_size=0.8,
    random_state=random_state,
    stratify=customer_data['target']
)
df_train.reset_index(drop=True, inplace=True)
df_test.reset_index(drop=True, inplace=True)
df_train['target'].value_counts(), df_test['target'].value_counts()

### Categorical values encoding using OneHotEncoder

In [ ]:
"""
Passing only the training data to the encoding function to avoid data leakage (target column is in numeric
form already at this stage so it shouldn't and won't be affected)
"""
df_train_encoded, ohe_fitted = hf.encode_categoric_data(df=df_train)
print()
df_test_encoded, _ = hf.encode_categoric_data(df=df_test, ohe=ohe_fitted)

### Correlation Matrix for continous variables

In [ ]:
df_train_encoded.head()

In [ ]:
flag_cols_mask = df_train_encoded.columns.str.contains('_flag')
numeric_corr = (
    df_train_encoded
    .loc[:, :'target']
	.corr()
    .round(decimals=2)
)
plt.figure(figsize=(20, 20))
sns.heatmap(
    numeric_corr,
    annot=True,
    cmap="coolwarm",
    vmin=-1,
    vmax=1
)
plt.show()

potential columns to reject because of big correlations:
- "monthly_payment",
- "tot_num_credit_lines"
- "paym_rec_for_tot_amnt_fund" or "princ_rec"

### VIF (Variance Inflation Factor) analysis to determine linearity degree between independent variables
$$ VIF_i = \frac{1}{1 - {R_i}^2} \;\;\;\;\;\;\; i = 1, 2 ... k $$

In [ ]:
vifs1 = hf.VIF( df_train_encoded.drop(columns='target') )
vifs1.iloc[:20, :]

In [ ]:
"""
Dropping columns that have VIF > 10 because the linearity among the predictors might significantly affect quality of the model,
also there are cases that VIF > 10 and the correlation with another variable based on the correlation matrix is very high so that
confirms the validity of dropping those columns
 - monthly_payment and funded_amnt are highly correlated and both have high VIF so dropping one of them
 - paym_rec_for_tot_amnt_fund and princ_rec are highly correlated and both have high VIF so dropping one of them,
 - loan_purpose_credit_card and loan_purpose_debt_consolidation both have high VIFs so dropping one of them
"""
cols_to_drop = ['monthly_payment', 'paym_rec_for_tot_amnt_fund', 'loan_purpose_debt_consolidation']
vifs2 = hf.VIF( df_train_encoded.drop(columns=[*cols_to_drop, 'target']) )
vifs2.iloc[:20, :]

In [ ]:
"""
Then also dropping the 'mths_sincerecent_revol_delinq' because of its second highest VIF that is probably caused by its artificial correlation
with 'mths_since_recent_revol_delinq_flag' (which actually has the greatest VIF but on the other hand the flag itself might be more significant
for prediction). Additionally based on the correlation matrix the 'mths_since_recent_revol_delinq' is also correlated (0,77) with 
'mths_since_recent_bankcard_delinq' so dropping one of the is a good step anyway.     
Generally at the same the purpose of those modifications was to stick to the rule to not exceed the established VIF threshold which is here assumed to be VIF* = 10. 
"""
cols_to_drop_v2 = [
    *cols_to_drop,
	'mths_since_recent_revol_delinq'
]
vifs3 = hf.VIF( df_train_encoded.drop(columns=[*cols_to_drop_v2, 'target']) ) 
vifs3.iloc[:20, :]

In [ ]:
top20_vifs = [vifs1.iloc[:20, :], vifs2.iloc[:20, :], vifs3.iloc[:20, :]]
all_vifs_dfs = []
vif_iters = []
for i, vif_df in enumerate(top20_vifs):
    vif_iters.append(f'Iter {i}')
    all_vifs_dfs.extend([
		vif_df.index,
        vif_df.values.ravel()
	])
df_vifs = pd.DataFrame(all_vifs_dfs).transpose()
# df_vifs.columns = 3*['Feature', 'VIF']
multi_cols = pd.MultiIndex.from_product([vif_iters, ['Feature', 'VIF']])
df_vifs.columns = multi_cols
df_vifs

In [ ]:
save_latex_table(
    df_vifs['Iter 0'], 
    cwd_rel_path='latex_tables/vifs0.tex', 
    include_index=False, 
    float_format="%.2f"
)
save_latex_table(
    df_vifs['Iter 1'], 
    cwd_rel_path='latex_tables/vifs1.tex', 
    include_index=False, 
    float_format="%.2f"
)
save_latex_table(
    df_vifs['Iter 2'], 
    cwd_rel_path='latex_tables/vifs2.tex', 
    include_index=False, 
    float_format="%.2f"
)

In [ ]:
df_train_encoded = df_train_encoded.drop( columns=cols_to_drop_v2 )
df_test_encoded = df_test_encoded.drop( columns=cols_to_drop_v2 )
df_train_encoded.shape, df_test_encoded.shape

In [ ]:
df_train_encoded.head()

In [ ]:
""" 
Cleaning feature names by replacing special characters with more suitable strings due to further
problems that occured for XGBoost with such names
"""
def clean_feature_names(df):
    df = df.copy()
    df.columns = (
        df.columns
        .astype(str)
        .str.replace('<', '_lt_', regex=False)
        .str.replace('>', '_gt_', regex=False)
        .str.replace('[', '_', regex=False)
        .str.replace(']', '_', regex=False)
        .str.replace('+', '_plus_', regex=False)
    )
    return df

df_train_encoded = clean_feature_names(df_train_encoded)
df_test_encoded = clean_feature_names(df_test_encoded)
df_train_encoded.head(10)

In [ ]:
df_train_encoded['target'].value_counts()

In [ ]:
X_train = df_train_encoded.drop(columns='target')
y_train = df_train_encoded['target']
X_test = df_test_encoded.drop(columns='target')
y_test = df_test_encoded['target']

#### Checking linearity assumptions for Logistic Regression (but it wasn't included in the end)

In [ ]:
# lin_check_res = hf.check_logit_linearity(X_cont=X_train.loc[:, :'credit_history_age'], y=y_train)

In [ ]:
# for key, value in lin_check_res.items():
# 	print(f"{key} | is linear: {value['is_linear']}")	

In [ ]:
# """Log(x + 1) transformation is applied to the features that do not have linear relationship with the logits of target variable"""
# non_lin_cols = [key for key, value in lin_check_res.items() if not value['is_linear']]
# non_lin_cols

In [ ]:
# """ 
# Applying log transformation to the features that do not have linear relationship with the logits of target variable, but 
# excluding 'num_30_plus__delinq_in_2yrs' and 'num_derogatory_pub_rec' because they are count variables and log transformation 
# would not be suitable for them
# """
# log_transform_cols = [col for col in non_lin_cols if col not in ['num_30_plus__delinq_in_2yrs', 'num_derogatory_pub_rec']]
# log_transform_cols

In [ ]:
# X_train[log_transform_cols] = X_train[log_transform_cols].apply(lambda x: np.log(x + 1))
# X_test[log_transform_cols] = X_test[log_transform_cols].apply(lambda x: np.log(x + 1))

In [ ]:
# lin_check_res_v2 = hf.check_logit_linearity(X_cont=X_train.loc[:, :'credit_history_age'], y=y_train)
# for key, value in lin_check_res_v2.items():
# 	print(f"{key} | is linear: {value['is_linear']}")

### Scaling numerical data

In [ ]:
""" 
Saving flag columns mask within numeric input features (columns) to filter them out before scaling, 
because they are binary variables. So we want to avoid unnecessary transformations.
"""
numeric_X_cols_flag_cols_mask = (
    X_train
    .loc[:, :'credit_history_age']
    .columns
    .str
    .endswith('_flag')
)
numeric_cols_to_transform = (
    X_train
    .loc[:, :'credit_history_age']
    .columns[~numeric_X_cols_flag_cols_mask]
)

numeric_X_cols_flag_cols_mask, len(numeric_X_cols_flag_cols_mask), numeric_cols_to_transform, len(numeric_cols_to_transform)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler
	
# Ultimately the RobustScaler was used to mitigate the influence of outliers, but other scalers were also tested
scaler = RobustScaler()

X_train[numeric_cols_to_transform] = scaler.fit_transform(
    X_train[numeric_cols_to_transform]
)
X_test[numeric_cols_to_transform] = scaler.transform(
    X_test[numeric_cols_to_transform]
)
X_train.head()

#### Using Lasso Logistic Regression for feature selection

In [ ]:
from sklearn.linear_model import LogisticRegressionCV

"""Using Lasso (L1) regularization with Logistic Regression to select features with non-zero coefficients"""
lasso_log_reg_start = time.time()
logrCV_model = LogisticRegressionCV(
    solver='saga',
    penalty='l1',
    class_weight='balanced',
    Cs=np.logspace(-4, 0, 20), # np.logspace(-4, 4, 40) was tested before but the optimal C was alwyas around 0.1 so the scope was limited for faster testing  
    scoring='roc_auc',
	max_iter=2000,
    tol=1e-4,
	cv=hf.StratifiedKFold(5, shuffle=True, random_state=random_state),
	random_state=random_state,
    n_jobs=-1
)
logrCV_model.fit(
    X=X_train,
    y=y_train
)
lasso_log_reg_end = time.time()
print(f"Lasso Logistic Regression training time: {(lasso_log_reg_end - lasso_log_reg_start):.2f} seconds")

col_names = X_train.columns
threshold_ = 1e-3
coeffs = logrCV_model.coef_.ravel()
selected_features_mask = np.abs(coeffs) > threshold_

# selected feature names
selected_features = col_names[selected_features_mask]
C = logrCV_model.C_[0]
print(f"C: {C:.3f}")
print(f"Lambda: {(1/C):.3f}")
print("\nFeatures with zeroed coeffs:")
print( *col_names[~selected_features_mask].to_list(), sep=",\n" )
print("\nSelected features:")
selected_features.to_list(), len(selected_features)

In [ ]:
lr_lasso_model_params = logrCV_model.get_params()
keys = list(lr_lasso_model_params.keys())
values = list(lr_lasso_model_params.values())

In [ ]:
keys

In [ ]:
# StratifiedKFold object couldn't be saved to latex table so needed to filter it out
not_cv = list(map(lambda x: x != 'cv', keys))
filt_keys = [keys[i] for i, mask in enumerate(not_cv) if mask == True]
filt_values = [values[i] for i, mask in enumerate(not_cv) if mask == True]
filt_values[0] = logrCV_model.C_[0]

In [ ]:
filt_keys, filt_values

In [ ]:
lr_lasso_params = pd.DataFrame(index=filt_keys, data=filt_values, columns=['Value'])
lr_lasso_params.index.name = 'Parameter - Lasso LR'
lr_lasso_params

In [ ]:
save_latex_table(
	df=lr_lasso_params, 
	cwd_rel_path='latex_tables/lasso_lr_params.tex', 
	include_index=True, 
	float_format="%0.2f"
)

In [ ]:
from itertools import batched

batches = list( batched(iterable=list(selected_features), n=21) )
df_selected_features = pd.DataFrame({'Feat1': batches[0], 'Feat2': batches[1]})
df_selected_features

In [ ]:
save_latex_table(
    df_selected_features, 
    cwd_rel_path='latex_tables/selected_features.tex',
    include_index=False
)

In [ ]:
""" Selecting only features with non-zero coefficients from Logistic Regression with L1 regularization """
""" Final X_train, X_test """
X_train = X_train[selected_features]
X_test = X_test[selected_features]

In [ ]:
X_train.head()

In [ ]:
# Numeric cols
num_cols = X_train.loc[:, :'log_late_fees_rec']
# OneHot Encoded columns
last_num_col_index = X_train.columns.to_list().index('log_late_fees_rec')
one_hot_encoded_cols = X_train.iloc[:, (last_num_col_index+1):]
num_cols.shape[1], one_hot_encoded_cols.shape[1]

### Logistic Regression Model
$$ p=P(Y=1|X)= \frac{1}{1+\exp{(-(\beta_0+\beta_1 X_1+...+\beta_k X_k))}}

In [ ]:
from sklearn.linear_model import LogisticRegression

beta = 2 # giving more weight to recall than precision

start_time = time.time()
log_reg_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    penalty='l2',
    random_state=random_state
)
lr_cv_res, lr_cv_agg_res, lr_beta = hf.model_cross_validation(
	log_reg_model,
	X_train,
	y_train,
	fbeta_param=beta,
	model_name='Logistic Regression',
	random_state=random_state,
	verbose=False
)
end_time = time.time()
print(f"Logistic Regression cross validation time: {(end_time - start_time):.2f} seconds")

In [ ]:
lr_model_params = log_reg_model.get_params()
lr_params = pd.DataFrame(index=lr_model_params.keys(), data=lr_model_params.values(), columns=['Value'])
lr_params.index.name = 'Parameter - LR'

save_latex_table(
	df=lr_params, 
	cwd_rel_path='latex_tables/logistic_regression_params.tex', 
	include_index=True, 
	float_format="%0.2f"
)

In [ ]:
old_format = pd.options.display.float_format

In [ ]:
pd.options.display.float_format = '{:.3f}'.format
lr_cv_res

In [ ]:
lr_cv_agg_res

In [ ]:
""" Save Logistic Regression cross validation results to latex tables """
save_latex_table(
    df=lr_cv_res, 
    cwd_rel_path='latex_tables/lr_cv_res.tex', 
    include_index=False, 
    float_format="%.3f"
)
save_latex_table(
    df=lr_cv_agg_res, 
    cwd_rel_path='latex_tables/lr_cv_agg_res.tex', 
    include_index=True, 
	float_format="%.3f"
)

In [ ]:
pd.options.display.float_format = old_format

In [ ]:
""" Training model on full training data """
log_reg_model.fit(
    X=X_train,
    y=y_train
)

In [ ]:
# """ Coefficients standard errors """
# from sklearn.base import clone
# from sklearn.utils import resample

# B = 2000
# rng = np.random.default_rng(random_state)

# bootstrap_coeffs = []

# for b in range(B):

#     # Bootstrap sample
#     indices = resample(
#         np.arange(len(y_train)),
#         replace=True,
#         n_samples=len(y_train),
#         random_state=int(rng.integers(0, 2**31 - 1))
#     )

#     X_boot = X_train.iloc[indices]
#     y_boot = y_train.iloc[indices]

#     # Fit the same model specification
#     model_boot = clone(log_reg_model)

#     model_boot.fit(X_boot, y_boot)

#     # Include intercept
#     coeffs = np.concatenate([
#         model_boot.intercept_,
#         model_boot.coef_.ravel()
#     ])

#     bootstrap_coeffs.append(coeffs)

# bootstrap_coefs = np.asarray(bootstrap_coeffs)

# # Standard errors
# standard_errors = bootstrap_coefs.std(
#     axis=0,
#     ddof=1
# )

# # Original model coefficients
# original_coefs = np.concatenate([
#     log_reg_model.intercept_,
#     log_reg_model.coef_.ravel()
# ])

# feature_names = [
#     'Intercept',
#     *X_train.columns
# ]

# results = pd.DataFrame({
#     'Feature': feature_names,
#     'Coefficient': original_coefs,
#     'Std. Error': standard_errors,
#     'Odds Ratio': np.exp(original_coefs),
#     'CI 2.5%': np.percentile(
#         bootstrap_coefs, 2.5, axis=0
#     ),
#     'CI 97.5%': np.percentile(
#         bootstrap_coefs, 97.5, axis=0
#     )
# })

# print(results)

In [ ]:
""" Logistic regression coefficients and their standard erros """

# Probabilities predicitions on train data
probs = log_reg_model.predict_proba(X_train)

# Design matrix - add column of 1's at the beginning of X_train matrix
X_design = np.hstack([np.ones((X_train.shape[0], 1)), X_train])

# Initiate matrix of 0's, fill diagonal with each predicted observation's variance =>
# p0 * p1 <=> p1 * (1 - p1), p1 - probability of belonging to class 1 
V = np.diagflat(np.product(probs, axis=1))
print(f'V matrix: \n {V}')

# Covariance matrix
# Note that the @-operater does matrix multiplication in Python 3.5+, so if you're running
# Python 3.5+, you can replace the covLogit-line below with the more readable:
# covLogit = np.linalg.inv(X_design.T @ V @ X_design)
covLogit = np.linalg.inv(X_design.T @ V @ X_design)
print(f'Covariance matrix: \n{covLogit}')

# Standard errors
std_errors = np.sqrt(np.diag(covLogit))
print(f'Standard errors: \n{std_errors}')

In [ ]:
""" Standard errors based on Sandwich covariance estimator """
from scipy.linalg import solve
from sklearn.utils.class_weight import compute_sample_weight

X_train_array = np.asarray(X_train)
y_train_array = np.asarray(y_train)

p = log_reg_model.predict_proba(X_train)[:, 1]
# Class weights
weights = compute_sample_weight(
    class_weight='balanced',
    y=y_train
)

# Weighted variance of Bernoulli predictions
v = weights * p * (1 - p)

# Hessian of weighted loss
H = X_design.T @ (X_design * v[:, None])

# L2 penalty (intercept is not penalized)
D = np.eye(X_design.shape[1])
D[0, 0] = 0

# Inverse Hessian
H_inv = solve(
    H,
    np.eye(H.shape[0]),
    assume_a='pos'
)

# Sandwich covariance estimator
scores = X_design * (
    weights * (y_train_array - p)
)[:, None]

J = scores.T @ scores

cov_sandwich = H_inv @ J @ H_inv

se_sandwich = np.sqrt(
    np.maximum(np.diag(cov_sandwich), 0)
)

In [ ]:
""" Model coeffs and their standard errors """
model_coeffs = np.concatenate([
    log_reg_model.intercept_,
    log_reg_model.coef_.flatten()
])

feature_names = [
    'Intercept',
    *X_train.columns
]

results = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': model_coeffs,
    'SE': std_errors,
    'SE_sandwich': se_sandwich
})

print(results)

In [ ]:
save_latex_table(
    results.loc[:, results.columns != 'SE'],
    'latex_tables/log_reg_coeffs_flat.tex',
    float_format='%.3f'
)

In [ ]:
""" Logistic Regression coeffs table in more compact form for documentation in LaTeX """
results_even_row_count = (
    pd.concat(
    [results, pd.DataFrame([[np.NaN, np.NaN, np.NaN, np.NaN]], columns=results.columns)], 
    axis=0)
    .reset_index(drop=True)
)
results_even_row_count.tail()

In [ ]:
feat_batches = list( batched(iterable=list(results_even_row_count.values), n=22) )
multi_columns = pd.MultiIndex.from_product([[1, 2], results_even_row_count.columns.tolist()])
lr_coeffs_df = pd.DataFrame(columns=multi_columns)
col_idx_map = {col: idx for idx, col in enumerate(results_even_row_count.columns)}
for mcol in lr_coeffs_df.columns:
	for i, batch_elem in enumerate(feat_batches[mcol[0]-1]):
		lr_coeffs_df.loc[i, mcol] = batch_elem[ col_idx_map[mcol[1]] ]
	
lr_coeffs_df

In [ ]:
save_latex_table(
    lr_coeffs_df,
    'latex_tables/log_reg_coeffs.tex',
    float_format='%.3f'
)

In [ ]:
y_probs_LR_train = log_reg_model.predict_proba(X_train)
y_probs_LR_test = log_reg_model.predict_proba(X_test)

# Obtaining best threshold for Logistic Regression based on Fbeta-score
lr_best_threshold = lr_cv_agg_res.loc['Threshold', 'Median']

y_pred_LR_train = np.where( y_probs_LR_train[:, 1] > lr_best_threshold, 1, 0 )
y_pred_LR_test = np.where( y_probs_LR_test[:, 1] > lr_best_threshold, 1, 0 ) # log_reg_model.predict(X_test)

#### Confusion Matrices

In [ ]:
cm_valid_LR, cm_test_LR = hf.display_confusion_matrix(
    y1_true=y_train,
	y1_pred=y_pred_LR_train,
	y2_true=y_test,
	y2_pred=y_pred_LR_test,
	# title='Confusion Matrices - Logistic Regression | Train vs Test',
	model_name='logistic regression',
	cmap='Blues',
	compare=['Train', 'Test']
)
hf.quality_metrics(
    y_true=y_train,
    y_pred=y_pred_LR_train,
    y_probs=y_probs_LR_train[:, 1],
    label='Train data',
    pr_curve_title='PR Curve Logistic Regression - Train data'
)
clf_report_lr_test = hf.quality_metrics(
    y_true=y_test,
	y_pred=y_pred_LR_test,
	y_probs=y_probs_LR_test[:, 1],
	label='\nTest data',
	pr_curve_title='PR Curve Logistic Regression - Test data'
)

In [ ]:
from string import capwords # Capitalizing first letter of each string

def transform_eval_res_dict_to_df(clf_report, y_true, y_pred, beta=2):
	# Input is classification_report from sklearn where accuracy is presented in columns while other quality measures are in rows so the
	# function transforms it so every measure name is in row and cols refer to class label
	df = clf_report.loc[['precision', 'recall', 'f1-score'], ['0', '1']]
	accuracy = clf_report['accuracy'].iloc[0]
	columns = pd.MultiIndex.from_product([['Class'], ['0', '1']])
	# Adding Fbeta-score to the table as well
	fbeta_score_neg = hf.fbeta_score(y_true=y_true, y_pred=y_pred, beta=beta, pos_label=0)
	fbeta_score_pos = hf.fbeta_score(y_true=y_true, y_pred=y_pred, beta=beta, pos_label=1)
	df.loc[f'f{beta}-score', '0'] = fbeta_score_neg
	df.loc[f'f{beta}-score', '1'] = fbeta_score_pos
	df.loc['accuracy', :] = accuracy
	df.index = df.index.map(capwords)
	df.columns = columns

	return df

lr_res_table_test = transform_eval_res_dict_to_df(
	clf_report=clf_report_lr_test, 
	y_true=y_test, 
	y_pred=y_pred_LR_test
)
lr_res_table_test

In [ ]:
save_latex_table(
	df=lr_res_table_test, 
	cwd_rel_path='latex_tables/lr_res_table_test.tex', 
	include_index=True, 
	float_format="%.3f"
)

#### Feature importances and SHAP values for Logistic Regression model

In [ ]:
""" Feature importance for Logistic Regression model """
lr_scorer = hf.make_threshold_scorer(
    scoring='fbeta',
    threshold=lr_best_threshold,
	beta=lr_beta
)
lr_feat_names_sorted_desc_by_importance = hf.plot_feature_importances(
    model=log_reg_model,
	model_name='Logistic Regression',
	X_data=X_train,
	y_data=y_train,
	n_reps=5,
	scoring=lr_scorer,
	max_num_features=15,
	n_jobs=-1
)

In [ ]:
hf.plot_shap_values(
    model=log_reg_model,
    model_name='Logistic Regression',
    X_train=X_train,
    y_train=y_train,
    X_valid=X_train,
    y_valid=y_train,
    threshold=lr_best_threshold,
    random_state=random_state
)

#### PCA Visualization - Logistic Regression

In [ ]:
colors_dict = {0: 'skyblue', 1: 'red'}
class_names_dict = {0: 'no default', 1: 'default'}
hf.pca_visualization(
    X_train=X_train,
    X_new=X_test,
    y_new_true=y_test,
    y_new_pred=y_pred_LR_test,
    colors_dict=colors_dict,
    class_names_dict=class_names_dict,
    model_name='Logistic Regression',
    valid_or_test='Test',
    alpha=0.4
)

### XGBoost algorithm

In [ ]:
import torch

""" setting device """
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

In [ ]:
import xgboost as xgb

val_counts = y_train.value_counts()
class_freq_ratio = val_counts.loc[0] / val_counts.loc[1]

start_time = time.time()
xgb_model = xgb.XGBClassifier(
    n_estimators=1000,
    learning_rate=0.01,
    min_child_weight=8,
    max_depth=4,
    objective='binary:logistic',
    subsample=0.8,
    # colsample_bytree=0.6,
    eval_metric='logloss',
    scale_pos_weight=class_freq_ratio,
    random_state=random_state
)
xgb_cv_res, xgb_cv_agg_res, xgb_beta = hf.model_cross_validation(
	xgb_model,
	X_train,
	y_train,
    fbeta_param=beta,
	model_name='XGBoost',
	random_state=random_state,
	verbose=False
)
end_time = time.time()
print(f"XGBoost cross validation time: {(end_time - start_time):.2f} seconds")

In [ ]:
xgb_model_params = xgb_model.get_params()
xgb_params = pd.DataFrame(index=xgb_model_params.keys(), data=xgb_model_params.values(), columns=['Value'])
xgb_params.index.name = 'Parameter - XGB'

save_latex_table(
	df=xgb_params, 
	cwd_rel_path='latex_tables/xgboost_params.tex', 
	include_index=True, 
	float_format="%0.2f"
)

In [ ]:
pd.options.display.float_format = '{:.3f}'.format
xgb_cv_res

In [ ]:
xgb_cv_agg_res

In [ ]:
""" Save XGBoost cross validation results to latex tables """
save_latex_table(df=xgb_cv_res, cwd_rel_path='latex_tables/xgb_cv_res.tex', include_index=False, float_format="%.3f")
save_latex_table(df=xgb_cv_agg_res, cwd_rel_path='latex_tables/xgb_cv_agg_res.tex', include_index=True, float_format="%.3f")

In [ ]:
pd.options.display.float_format = old_format

In [ ]:
X_train_xgb, X_valid_xgb, y_train_xgb, y_valid_xgb = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    stratify=y_train
)
# Setting early stopping for fitting final model, in the cross validation there was no early stopping included
xgb_model.early_stopping_rounds = 50
xgb_model.fit(
    X=X_train_xgb,
    y=y_train_xgb,
    eval_set=[(X_valid_xgb, y_valid_xgb)],
    verbose=50
)

In [ ]:
# Probabilities
y_probs_XGB_train = xgb_model.predict_proba(X_train_xgb)
y_probs_XGB_test = xgb_model.predict_proba(X_test)

# Obtaining best threshold for XGBoost based on recall
xgb_best_threshold = xgb_cv_agg_res.loc['Threshold', 'Median']

# Predictions (labels)
y_pred_XGB_train = np.where(y_probs_XGB_train[:, 1] > xgb_best_threshold, 1, 0) # xgb_model.predict(X_train)
y_pred_XGB_test = np.where(y_probs_XGB_test[:, 1] > xgb_best_threshold, 1, 0) # xgb_model.predict(X_test)

In [ ]:
cm_valid_XGB, cm_test_XGB = hf.display_confusion_matrix(
    y1_true=y_train_xgb,
    y1_pred=y_pred_XGB_train,
    y2_true=y_test,
    y2_pred=y_pred_XGB_test,
    # title='Confusion Matrices - XGBoost | Train vs Test',
    model_name='xgboost',
    cmap="Blues",
    compare=['Train', 'Test']
)
hf.quality_metrics(
    y_true=y_train_xgb,
    y_pred=y_pred_XGB_train,
    y_probs=y_probs_XGB_train[:, 1],
    label='Train data',
    pr_curve_title='PR Curve XGBoost - Train data'
)
clf_report_xgb_test = hf.quality_metrics(
    y_true=y_test,
    y_pred=y_pred_XGB_test,
    y_probs=y_probs_XGB_test[:, 1],
    label='\nTest data',
    pr_curve_title='PR Curve XGBoost - Test data'
)

In [ ]:
xgb_res_table_test = transform_eval_res_dict_to_df(
    clf_report=clf_report_xgb_test,
    y_true=y_test,
	y_pred=y_pred_XGB_test
)
xgb_res_table_test

In [ ]:
save_latex_table(
	df=xgb_res_table_test, 
	cwd_rel_path='latex_tables/xgb_res_table_test.tex', 
	include_index=True, 
	float_format="%.3f"
)

#### Visualizing decision trees, feature importances and SHAP values for XGBoost model

In [ ]:
""" Plots decision trees for XGBoost model for chosen tree indices (first and last tree) """
tree_indices = [0, xgb_model.best_iteration]
fig, axes = plt.subplots(1, len(tree_indices), figsize=(60, 80))
axes = axes.flatten()
for tree_i, ax in zip(tree_indices, axes):
    xgb.plot_tree(
        booster=xgb_model,
        tree_idx=tree_i,
        ax=ax
    )
    ax.set_title(f'Tree number: {tree_i}')
plt.tight_layout()
plt.show()

In [ ]:
""" Feature importances for XGBoost model """
xgb_scorer = hf.make_threshold_scorer(
    scoring='fbeta',
    threshold=xgb_best_threshold,
    beta=xgb_beta
)
xgb_feat_names_sorted_desc_by_importance = hf.plot_feature_importances(
    model=xgb_model,
    model_name='XGBoost',
    X_data=X_train_xgb,
    y_data=y_train_xgb,
    n_reps=5,
    scoring=xgb_scorer,
    max_num_features=15,
    n_jobs=-1
)

In [ ]:
hf.plot_shap_values(
    model=xgb_model,
    model_name='XGBoost',
    X_train=X_train_xgb,
    y_train=y_train_xgb,
    X_valid=X_train_xgb,
    y_valid=y_train_xgb,
    threshold=xgb_best_threshold,
    random_state=random_state
)

#### PCA Visualization - XGBoost

In [ ]:
hf.pca_visualization(
    X_train=X_train_xgb,
    X_new=X_test,
    y_new_true=y_test,
    y_new_pred=y_pred_XGB_test,
    colors_dict=colors_dict,
    class_names_dict=class_names_dict,
    model_name='XGBoost',
    valid_or_test='Test',
	alpha=0.4
)

### Random Forest Classifier

In [ ]:
val_counts

In [ ]:
from sklearn.ensemble import RandomForestClassifier

start_time = time.time()
rf_model = RandomForestClassifier(
    n_estimators=200,
    criterion='gini',
    class_weight='balanced_subsample',
    max_depth=20,
    min_samples_leaf=4,
    random_state=random_state,
    n_jobs=-1
)
rf_cv_res, rf_cv_agg_res, rf_beta = hf.model_cross_validation(
	rf_model,
	X_train,
	y_train,
    fbeta_param=beta,
	model_name='Random Forest',
	random_state=random_state,
	verbose=False
)
end_time = time.time()
print(f"Random Forest cross validation time: {(end_time - start_time):.2f} seconds")

In [ ]:
rf_model_params = rf_model.get_params()
rf_params = pd.DataFrame(index=rf_model_params.keys(), data=rf_model_params.values(), columns=['Value'])
rf_params.index.name = 'Parameter - RF'

save_latex_table(
	df=rf_params, 
	cwd_rel_path='latex_tables/random_forest_params.tex', 
	include_index=True, 
	float_format="%0.2f"
)

In [ ]:
pd.options.display.float_format = '{:.3f}'.format
rf_cv_res

In [ ]:
rf_cv_agg_res

In [ ]:
""" Save Random Forest cross validation results to latex tables """
save_latex_table(
    df=rf_cv_res, 
    cwd_rel_path='latex_tables/rf_cv_res.tex',
    include_index=False, 
    float_format="%.3f"
)
save_latex_table(
    df=rf_cv_agg_res, 
    cwd_rel_path='latex_tables/rf_cv_agg_res.tex', 
    include_index=True, 
    float_format="%.3f"
)

In [ ]:
pd.options.display.float_format = old_format

In [ ]:
rf_model.fit(
    X=X_train,
    y=y_train
)

In [ ]:
y_probs_RF_train = rf_model.predict_proba(X_train)
y_probs_RF_test = rf_model.predict_proba(X_test)

# Obtaining best threshold for Random Forest based on recall
rf_best_threshold = rf_cv_agg_res.loc['Threshold', 'Median']

# Predictions (labels)
y_pred_RF_train = np.where(y_probs_RF_train[:, 1] > rf_best_threshold, 1, 0) # rf_model.predict(X_train)
y_pred_RF_test = np.where(y_probs_RF_test[:, 1] > rf_best_threshold, 1, 0) # rf_model.predict(X_test)

In [ ]:
cm_train_RF, cm_test_RF = hf.display_confusion_matrix(
    y1_true=y_train,
	y1_pred=y_pred_RF_train,
	y2_true=y_test,
	y2_pred=y_pred_RF_test,
	# title='Confusion Matrices - Random Forest | Train vs Test',
	model_name='random forest',
	cmap="Blues",
	compare=['Train', 'Test']
)
hf.quality_metrics(
    y_true=y_train,
	y_pred=y_pred_RF_train,
	y_probs=y_probs_RF_train[:, 1],
	label='Train data',
	pr_curve_title='PR Curve Random Forest - Train data'
)
clf_report_rf_test = hf.quality_metrics(
    y_true=y_test,
	y_pred=y_pred_RF_test,
	y_probs=y_probs_RF_test[:, 1],
	label='\nTest data',
	pr_curve_title='PR Curve Random Forest - Test data'
)

In [ ]:
rf_res_table_test = transform_eval_res_dict_to_df(
    clf_report=clf_report_rf_test,
    y_true=y_test,
	y_pred=y_pred_RF_test
)
rf_res_table_test

In [ ]:
save_latex_table(
    df=rf_res_table_test, 
	cwd_rel_path='latex_tables/rf_res_table_test.tex', 
	include_index=True, 
	float_format="%.3f"
)

#### Visualizing decision trees, feature importances and SHAP values for Random Forest model

In [ ]:
from sklearn import tree

""" Plots decision trees for Random Forest model for chosen tree indices (first and last tree) """
estimators_to_plot = [0, -1] # first and last trees in the forest
fig, axes = plt.subplots(len(estimators_to_plot), 1, figsize=(15, 15))
axes = axes.flatten()
max_depth = 8
for tree_i, ax in zip(estimators_to_plot, axes):
    tree.plot_tree(
		decision_tree=rf_model.estimators_[tree_i],
		feature_names=rf_model.feature_names_in_,
		class_names=rf_model.classes_.astype(str),
		filled=True,
		rounded=True,
		ax=ax,
		max_depth=max_depth
    )
plt.tight_layout()
plt.show()

In [ ]:
"""Feature importances for Random Forest model"""
rf_scorer = hf.make_threshold_scorer(
    scoring='fbeta',
	threshold=rf_best_threshold,
	beta=rf_beta
)
rf_feat_names_sorted_desc_by_importance = hf.plot_feature_importances(
	model=rf_model,
	model_name='Random Forest',
	X_data=X_train,
	y_data=y_train,
	n_reps=5,
	scoring=rf_scorer,
	max_num_features=15,
	n_jobs=1
)

In [ ]:
hf.plot_shap_values(
	model=rf_model,
	model_name='Random Forest',
	X_train=X_train,
	y_train=y_train,
	X_valid=X_train,
	y_valid=y_train,
	threshold=rf_best_threshold,
	random_state=random_state
)

#### PCA Visualization - Random Forest

In [ ]:
hf.pca_visualization(
	X_train=X_train,
	X_new=X_test,
	y_new_true=y_test,
	y_new_pred=y_pred_RF_test,
	colors_dict=colors_dict,
	class_names_dict=class_names_dict,
	model_name='Random Forest',
	valid_or_test='Test',
	alpha=0.4
)

### Neural Network

Training Neural Network model and validation on validation data

In [ ]:
from nn_for_classification import NeuralNetClassifier
from sklearn.utils.class_weight import compute_class_weight

# Computing class weights to handle class imbalance in the target variable for Neural Network model
class_weights = compute_class_weight(
	class_weight='balanced',
	classes=np.unique(y_train),
	y=y_train
)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

start_time = time.time()
nn_model = NeuralNetClassifier(
	hidden_layers_sizes=[64],
	dropout_prob=0.2,
	class_weights=class_weights,
	learning_rate=0.001,
	validation_split=0.2,
	epochs=2000,
	regularization_type='L2',
	lambda_reg=0.001,
	patience=50,
	random_state=random_state,
	device=device
)
nn_cv_res, nn_cv_agg_res, nn_beta = hf.model_cross_validation(
	nn_model,
	X_train,
	y_train,
    fbeta_param=beta,
	model_name='Neural Network',
	random_state=random_state,
	verbose=False
)
end_time = time.time()
print(f"\nNeural network cross validation time: {(end_time - start_time):.2f} seconds")

In [ ]:
print(class_weights[1] / class_weights[0])

In [ ]:
nn_model_params = nn_model.get_params()
nn_model_params

In [ ]:
# Converting class_weights to list to make it possible to read 
nn_model_params['class_weights'] = nn_model_params['class_weights'].tolist()

In [ ]:
nn_params = pd.DataFrame(index=list(nn_model_params.keys())[1:], data=list(nn_model_params.values())[1:], columns=['Value'])
nn_params.index.name = 'Parameter - NN'

save_latex_table(
	df=nn_params, 
	cwd_rel_path='latex_tables/neural_network_params.tex', 
	include_index=True, 
	float_format="%0.3f"
)

In [ ]:
pd.options.display.float_format = '{:.3f}'.format
nn_cv_res

In [ ]:
nn_cv_agg_res

In [ ]:
""" Save Neural Network cross validation results to latex tables """
save_latex_table(
    df=nn_cv_res, 
    cwd_rel_path='latex_tables/nn_cv_res.tex', 
    include_index=False, 
    float_format="%.3f"
)
save_latex_table(
	df=nn_cv_agg_res, 
	cwd_rel_path='latex_tables/nn_cv_agg_res.tex', 
	include_index=True, 
	float_format="%.3f"
)

In [ ]:
pd.options.display.float_format = old_format

In [ ]:
""" 
Neural Network model is trained on the whole training set with validation split inside the fit method for early stopping purposes
"""
nn_model.fit(
    X=X_train,
    y=y_train
)

#### Training process visualisation

In [ ]:
history = nn_model.history
epochs_axis = range(1, history.break_epoch + 1)
plt.figure(figsize=(13, 5))
plt.suptitle('Neural network training process', fontsize=14)
plt.subplot(1, 2, 1)
plt.plot(
    epochs_axis,
    history.train_losses,
    color='blue',
    linewidth=1,
    label='Train loss',
)
plt.plot(
    epochs_axis,
    history.valid_losses,
    color='red',
    linewidth=1,
    label='Valid loss'
)
plt.axvline(
    x=history.best_epoch,
    color='grey',
    linewidth=1.5,
    linestyle='--',
    label=f'Best result epoch = {history.best_epoch}'
)
plt.axvline(
    x=history.break_epoch,
    color='black',
    linewidth=1.5,
    linestyle='--',
    label=f'Last epoch = {history.break_epoch}'
)
plt.legend(fontsize=12, loc='upper right')
plt.xlabel('Epoch', fontsize=12); plt.ylabel('Loss', fontsize=12)
plt.grid(linewidth=0.4)
plt.tight_layout()

plt.subplot(1, 2, 2)
plt.plot(
    epochs_axis,
    history.train_pr_aucs,
    color='blue',
    linewidth=1,
    label='Train PR AUC',
)
plt.plot(
    epochs_axis,
    history.valid_pr_aucs,
    color='red',
    linewidth=1,
    label='Valid PR AUC'
)
plt.axvline(
    x=history.best_epoch,
    color='grey',
    linewidth=1.5,
    linestyle='--',
    label=f'Best result epoch = {history.best_epoch}'
)
plt.axvline(
    x=history.break_epoch,
    color='black',
    linewidth=1.5,
    linestyle='--',
    label=f'Last epoch = {history.break_epoch}'
)
plt.legend(fontsize=12, loc='lower right')
plt.xlabel('Epoch', fontsize=12); plt.ylabel('PR AUC', fontsize=12)
plt.grid(linewidth=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# Probabilities
y_probs_NN_train = nn_model.predict_proba(X_train)

# Obtaining best threshold for Neural Network based on Fbeta-score
nn_best_threshold = nn_cv_agg_res.loc['Threshold', 'Median']

# Predictions on train
y_pred_NN_train = np.where(y_probs_NN_train[:, 1] > nn_best_threshold, 1, 0) # nn_model.predict(X_train)

#### Testing model on test data

In [ ]:
""" Evaluating best model on test data """
best_model = nn_model.history.best_model
print(best_model.layer_stack)

""" predict_proba() method uses best model """
y_probs_NN_test = nn_model.predict_proba(X_test)
y_pred_NN_test = np.where(y_probs_NN_test[:, 1] > nn_best_threshold, 1, 0) # nn_model.predict(X_test)

# Loss on test data
X_test_tensor = torch.tensor(X_test.to_numpy(), dtype=torch.float32, device=device)
y_test_tensor = torch.tensor(y_test.to_numpy(), dtype=torch.long, device=device)
y_test_logits = best_model(X_test_tensor).squeeze() # forward step to get logits for test data
test_loss = history.loss_fn(y_test_logits, y_test_tensor).item()
print(f"Test loss: {test_loss:.5f}")

In [ ]:
cm_train_NN, cm_test_NN = hf.display_confusion_matrix(
    y1_true=y_train,
    y1_pred=y_pred_NN_train,
    y2_true=y_test,
    y2_pred=y_pred_NN_test,
    # title='Confusion Matrices - Neural Network | Train vs Test',
    model_name='neural network',
    cmap="Blues",
    compare=['Train', 'Test']
)
hf.quality_metrics(
    y_true=y_train,
    y_pred=y_pred_NN_train,
    y_probs=y_probs_NN_train[:, 1],
    label='Train data',
    pr_curve_title='PR Curve Neural Network - Train data'
)
clf_report_nn_test = hf.quality_metrics(
    y_true=y_test,
    y_pred=y_pred_NN_test,
    y_probs=y_probs_NN_test[:, 1],
    label='\nTest data',
    pr_curve_title='PR Curve Neural Network - Test data'
)

In [ ]:
nn_res_table_test = transform_eval_res_dict_to_df(
    clf_report=clf_report_nn_test,
	y_true=y_test,
	y_pred=y_pred_NN_test
)
nn_res_table_test

In [ ]:
save_latex_table(
    df=nn_res_table_test, 
	cwd_rel_path='latex_tables/nn_res_table_test.tex', 
	include_index=True, 
	float_format="%.3f"
)

#### Feature importances and SHAP values for Neural Network model

In [ ]:
nn_scorer = hf.make_threshold_scorer(
    scoring='fbeta',
	threshold=nn_best_threshold,
    beta=nn_beta
)
nn_feat_names_sorted_desc_by_importance = hf.plot_feature_importances(
	model=nn_model,
	model_name='Neural Network',
	X_data=X_train,
	y_data=y_train,
	n_reps=5,
	scoring=nn_scorer,
	max_num_features=15,
	n_jobs=2
)

In [ ]:
hf.plot_shap_values(
    model=nn_model,
    model_name="Neural Network",
    X_train=X_train,
    y_train=y_train,
	X_valid=X_train,
    y_valid=y_train,
    threshold=nn_best_threshold,
    random_state=random_state
)

#### PCA Visualization - Neural Network

In [ ]:
hf.pca_visualization(
	X_train=X_train,
	X_new=X_test,
	y_new_true=y_test,
	y_new_pred=y_pred_NN_test,
	colors_dict=colors_dict,
	class_names_dict=class_names_dict,
	model_name='Neural Network',
	valid_or_test='Test',
	alpha=0.4
)

### Visualising threshold values in each fold and for each method 

In [ ]:
df_thresholds = pd.DataFrame({
    'Fold': lr_cv_res['Fold'],
    'LR': lr_cv_res['Threshold'],
	'XGB': xgb_cv_res['Threshold'],
	'RF': rf_cv_res['Threshold'],
	'NN': nn_cv_res['Threshold']
})
df_thresholds_melted = df_thresholds.melt(
    id_vars='Fold', 
	var_name='Model', 
	value_name='Threshold'
)

_, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.barplot(
    data=df_thresholds_melted,
    x='Model',
    y='Threshold',
    hue='Fold',
    ax=axes[0],
    palette='Paired',
    edgecolor='black',
    width=0.65,
    alpha=0.8
)
axes[0].set_title('Thresholds distribution across folds for each model', fontsize=10)
sns.boxplot(
    data=df_thresholds_melted,
    x='Model',
	y='Threshold', 
    ax=axes[1],
    color='silver',
    width=0.5,
    medianprops={'color': 'red', 'linewidth': 0.6}
)
axes[1].set_title('Thresholds boxplots across folds for each model', fontsize=10)
plt.show()

### Visualising PR curves for each method for test data

In [ ]:
y_probs_test = [y_probs_LR_test, y_probs_XGB_test, y_probs_RF_test, y_probs_NN_test]
model_names = ['LR', 'XGB', 'RF', 'NN']
colors = ['dodgerblue', 'red', 'green', 'black']

plt.figure(figsize=(6.5, 5.5))
for probs, name, color in zip(y_probs_test, model_names, colors):
    precisions, recalls, _ = hf.precision_recall_curve(y_true=y_test, y_score=probs[:, 1], pos_label=1)
    avg_precision_score = hf.average_precision_score(y_true=y_test, y_score=probs[:, 1])
    plt.plot(
		recalls,
		precisions,
		label=f'{name} | AP = {avg_precision_score:.4f}',
		color=color
  	)
plt.xlabel('Recall', fontsize=12); plt.ylabel('Precision', fontsize=12)
plt.legend(loc='center left', fontsize=12)
plt.title('PR Curves Comparison of Models - Test data', fontsize=12)
plt.grid(linewidth=0.2)
plt.tight_layout()
plt.show()

In [ ]:
""" 
Fbeta-score for all models on test data with beta value calculated based on class imbalance 
in the target variable to give more weight to recall than precision
"""
lr_fbeta_score = hf.fbeta_score(y_true=y_test, y_pred=y_pred_LR_test, beta=beta)
xgb_fbeta_score = hf.fbeta_score(y_true=y_test, y_pred=y_pred_XGB_test, beta=beta)
rf_fbeta_score = hf.fbeta_score(y_true=y_test, y_pred=y_pred_RF_test, beta=beta)
nn_fbeta_score = hf.fbeta_score(y_true=y_test, y_pred=y_pred_NN_test, beta=beta)

lr_eval_scores = clf_report_lr_test.loc[['precision', 'recall', 'f1-score'], '1']
xgb_eval_scores = clf_report_xgb_test.loc[['precision', 'recall', 'f1-score'], '1']
rf_eval_scores = clf_report_rf_test.loc[['precision', 'recall', 'f1-score'], '1']
nn_eval_scores = clf_report_nn_test.loc[['precision', 'recall', 'f1-score'], '1']

lr_acc_score = clf_report_lr_test['accuracy'].iloc[0]
xgb_acc_score = clf_report_xgb_test['accuracy'].iloc[0]
rf_acc_score = clf_report_rf_test['accuracy'].iloc[0]
nn_acc_score = clf_report_nn_test['accuracy'].iloc[0]

models_eval_res = {
    "LR": [ *lr_eval_scores.values, lr_fbeta_score, lr_acc_score ],
    "XGB": [ *xgb_eval_scores.values, xgb_fbeta_score, xgb_acc_score ],
    "RF": [ *rf_eval_scores.values, rf_fbeta_score, rf_acc_score ],
    "NN": [ *nn_eval_scores.values, nn_fbeta_score, nn_acc_score ]
}
fbeta_score_colname = f'F{beta}-score'

In [ ]:
old_format = pd.options.display.float_format
pd.options.display.float_format = '{:.3f}'.format

# Scores for positive class for each model on test data in dataframe
model_scores_df = pd.DataFrame(
    models_eval_res.values(),
    index=models_eval_res.keys(),
    columns=[ *lr_eval_scores.index.map(capwords), fbeta_score_colname, 'Accuracy' ]
)
model_scores_df

In [ ]:
save_latex_table(
	df=model_scores_df, 
	cwd_rel_path='latex_tables/model_scores_df.tex', 
	include_index=True, 
	float_format="%.3f"
)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
sns.barplot(
    data=model_scores_df,
    x="Precision",
    y=model_scores_df.index,
    ax=axes[0],
    linewidth=1,
    edgecolor='black',
    height=0.6,
    color='skyblue'
)
axes[0].bar_label(axes[0].containers[0], fmt='%.3f')
axes[0].set_xlabel("Precision")
axes[0].set_ylabel("Model")
axes[0].set_xlim( left=model_scores_df['Precision'].min() - 0.1 )

sns.barplot(
    data=model_scores_df,
    x="Recall",
    y=model_scores_df.index,
    ax=axes[1],
    linewidth=1,
    edgecolor='black',
    height=0.6,
    color='lightcoral'
)
axes[1].bar_label(axes[1].containers[0], fmt='%.3f')
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Model")
axes[1].set_xlim( left=model_scores_df['Recall'].min() - 0.1 )

sns.barplot(
    data=model_scores_df,
    x="F1-score",
    y=model_scores_df.index,
    ax=axes[2],
    linewidth=1,
    edgecolor='black',
    height=0.6,
    color='lightgreen'
)
axes[2].bar_label(axes[2].containers[0], fmt='%.3f')
axes[2].set_xlabel("F1-score")
axes[2].set_ylabel("Model")
axes[2].set_xlim( left=model_scores_df['F1-score'].min() - 0.1 )

sns.barplot(
    data=model_scores_df,
    x=fbeta_score_colname,
    y=model_scores_df.index,
    ax=axes[3],
    linewidth=1,
    edgecolor='black',
    height=0.6,
    color='silver'
)
axes[3].bar_label(axes[3].containers[0], fmt='%.3f')
axes[3].set_xlabel(fbeta_score_colname)
axes[3].set_ylabel("Model")
axes[3].set_xlim( left=model_scores_df[fbeta_score_colname].min() - 0.1 )
plt.suptitle('Evaluation metrics comparison for models - Test Data', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
pd.options.display.float_format = old_format
gloabl_end_time = time.time()
print(f"Total execution time: {(gloabl_end_time - global_start_time):.2f} seconds")